# task_cote_divoire — baseline walkthrough

Loads the two handed-out checkpoints and runs them against the public
`dataset/train-input.json` (covers both tasks) to show the symptom. This
notebook does **not** implement a fix or reveal the flag — it demonstrates
the I/O contract and why neither shipped checkpoint is a valid submission
on its own.

In [1]:
import json
import model_utils as mu

vectorizer = mu.load_vectorizer()
original = mu.load_model(mu.ORIGINAL_MODEL_PATH)
finetuned = mu.load_model(mu.FINETUNED_MODEL_PATH)

train_in = {r["id"]: r["text"] for r in mu.load_records("dataset/train-input.json")}
train_out = {r["id"]: r["label"] for r in mu.load_records("dataset/train-output.json")}
ids = list(train_in.keys())
texts = [train_in[i] for i in ids]
print(f"{len(ids)} public rows covering both tasks")

250 public rows covering both tasks


## The original checkpoint — never saw the new task

In [2]:
preds = mu.predict_labels(original, vectorizer, texts)
acc = mu.accuracy_by_task(train_out, dict(zip(ids, preds)))
print("original checkpoint:", acc)

original checkpoint: {'old': 1.0, 'new': 0.0}


## The fine-tuned checkpoint — this is what you are handed as your starting point

In [3]:
preds = mu.predict_labels(finetuned, vectorizer, texts)
acc = mu.accuracy_by_task(train_out, dict(zip(ids, preds)))
print("finetuned checkpoint:", acc)

finetuned checkpoint: {'old': 0.0, 'new': 1.0}


## The symptom

The original checkpoint aces the old task and can't do the new task at all
(it was never trained on it). The fine-tuned checkpoint aces the new task
and can't do the old task at all — not "worse", **zero**.

Neither checkpoint, used as-is, clears both of `evaluation_script.py`'s
accuracy gates (0.85 on each task, independently). Your job is to produce a
third checkpoint that does — see `task_description.md` for the exact
contract and `main_template.py` for the I/O shape your submission must
match. `dataset/old_task_sample.json` is a small sample of the original
task's data, given to you specifically for recovery.

## I/O contract — what your submission must do

In [4]:
# main_template.py defaults to checkpoints/recovered_model.joblib (the
# checkpoint you are expected to produce), which does not exist yet in this
# demo, so pass --model explicitly to try a shipped checkpoint and see the
# I/O contract work end to end: read an input file by filename only, write
# predictions by filename only.
import subprocess
result = subprocess.run(
    ["python", "main_template.py", "--input", "dataset/train-input.json",
     "--output", "demo_predictions.json",
     "--model", "checkpoints/finetuned_model.joblib"],
    capture_output=True, text=True,
)
print(result.stdout)
print(result.stderr)

wrote 250 predictions to demo_predictions.json




In [5]:
preds = mu.load_records("demo_predictions.json")
print(preds[:3])
import os
os.remove("demo_predictions.json")

[{'id': 'train-0000', 'label': 'NEW_WINDOW_LAG_DELIVERY_DELAY'}, {'id': 'train-0001', 'label': 'NEW_SPATIAL_WITHIN_RADIUS'}, {'id': 'train-0002', 'label': 'NEW_WINDOW_PARTITION_RANK_FLEET'}]
